In [ ]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy


In [ ]:
np.random.seed(0)

In [ ]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [ ]:
def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost + (i*1e-6))
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities + np.array([i*1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [ ]:
def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

In [ ]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.0
    for partition in partitions:
        acc_loss_p  = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [ ]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [ ]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))
    best_partition, best_loss = None, np.inf
    for partitions in partitions_set:
        acc_loss = evaluate_system(X, partitions, thresholds, priors, threshold_true, c)
        if acc_loss < best_loss:
            best_loss      = acc_loss
            best_partition = partitions
    return best_partition

In [ ]:
def display_priority_queue(pq, P):
    res = "[  "
    for acc_loss, (a_id, b_id) in pq:
        res += f"({acc_loss}, ({P[a_id]}, {P[b_id]}))  "
    res += "]"
    print(res)

def find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1
    
    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c) * np.sum(priors[a])
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c) * np.sum(priors[b])
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        if display:
            display_priority_queue(pq, P)
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        ab = sorted(a + b)
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])
        # print(ab,  lhs, rhs)

        if lhs - rhs > -1e-9:
            del P[a_id]
            del P[b_id]

            pq2 = []
            for acc_loss, (x_id, y_id) in pq:
                if x_id not in {a_id, b_id} and y_id not in {a_id, b_id}:
                    heapq.heappush(pq2, (acc_loss, (x_id, y_id)))
            pq = deepcopy(pq2)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    p = P[p_id]
                    merged = sorted(ab + p)
                    acc_loss_merged = evaluate_partition(X, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged])
                    acc_loss_p = evaluate_partition(X, p, thresholds, priors, threshold_true, c) * np.sum(priors[p])
                    acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
                    gain = -(acc_loss_p + acc_loss_ab - acc_loss_merged)
                    heapq.heappush(pq, (gain, (new_id, p_id)))
    return list(P.values())

In [ ]:
def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [ ]:
x_min, x_max, x_disc = 0., 1., 1e-4
X = np.arange(x_min, x_max+x_disc, x_disc).round(4)

In [1]:
df = pd.read_pickle("../results/grid_search_approx_ratio_best_n4.pkl")

NameError: name 'pd' is not defined

In [ ]:
df.head()

In [ ]:
df[df["r_mult"]>2.5]

In [ ]:
i = df["r_mult"].argmax()
df.iloc[[i]]

In [ ]:
df["opt_len"] = df["partition_opt"].apply(lambda x: len(x))
df["greedy_len"] = df["partition_greedy"].apply(lambda x: len(x))

In [ ]:
df_bad = df
df_bad.sort_values("r_mult", ascending=False)

In [ ]:
i = df_bad["r_mult"].argmax()
thresholds = df_bad["thresholds"].iloc[i]
priors = df_bad["priors"].iloc[i]
threshold_true = df_bad["threshold_true"].iloc[i]
c = df_bad["c"].iloc[i]
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

In [ ]:
display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.4f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r}")
print("-"*80)
print()

Can we find a situation where the optimal is big partition, and see what greedy does. Does greedy fail in such situations?

In [ ]:
a, b = [2], [3]

acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
rhs = acc_loss_ab * np.sum(priors[ab])
gain = lhs - rhs

print(f"    threshold true: {threshold_true:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"              gain: {gain:.7f}")
print(f"            merge?: {lhs - rhs > -1e-6}")
print()

In [ ]:
X_p = best_response_vectorized(X, thresholds[[0,2,3]], priors[[0,2,3]], c)
px.scatter(x=X, y=X_p)

Can this cause an issue in greedy? Make sure no issues due to lack of tie breaking. 

In [ ]:
epsilons = np.arange(0, 0.02, 1e-4)

c = 0.75
ratios = []
for eps in tqdm.tqdm(epsilons):
    priors = priors = np.array([0.32/(1-eps), 0.02-eps, 0.36/(1-eps), 0.3/(1-eps)])
    # print(np.sum(priors))
    partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    ratios.append(r)

In [ ]:
px.line(y=ratios, x=epsilons)

In [ ]:
thresholds = np.array([0., 0.2, 0.6, 1.0])
# priors = np.array([0.32, 0.02, 0.36, 0.3])
eps = epsilons[np.argmax(ratios)]
priors = np.array([0.32/(1-eps), 0.02-eps, 0.36/(1-eps), 0.3/(1-eps)])
print(np.sum(priors))
c = 0.75
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.6f}")
print(f"eps:           : {eps:.5f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r:.4f}")
print("-"*80)
print()

In [ ]:
a, b = [1], [3]

acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
rhs = acc_loss_ab * np.sum(priors[ab])
gain = lhs - rhs

print(f"    threshold true: {threshold_true:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs}")
print(f"               RHS: {rhs}")
print(f"              gain: {gain}")
print(f"            merge?: {lhs - rhs > -1e-6}")

In [ ]:
threshold_trues = np.arange(0.09, 0.11, 1e-5)
approx_ratios = []
for threshold_true in tqdm.tqdm(threshold_trues):
    partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    approx_ratios.append(r)

In [ ]:
np.max(approx_ratios), threshold_trues[np.argmax(approx_ratios)]

In [ ]:
px.scatter(y=approx_ratios, x=threshold_trues)

In [ ]:
thresholds = np.array([0., 0.2, 0.6, 1.0])
# priors = np.array([0.32, 0.02, 0.36, 0.3])
eps = epsilons[np.argmax(ratios)]
# eps = 0.02
left = eps/3
priors = np.array([0.32+eps/3, 0.02-eps, 0.36+2*eps/3, 0.3])
print(np.sum(priors))
threshold_true = threshold_trues[np.argmax(approx_ratios)]
c = 0.75
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.4f}")
print(f"eps:           : {eps:.5f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (benc)       : {2.8936:.4f}")
print(f"r (mult)       : {r:.4f}")
print("-"*80)
print()

In [ ]:
approx_ratios = np.array(approx_ratios)
t_star = threshold_trues[approx_ratios!=1][0]

In [ ]:
thresholds = np.array([0., 0.2, 0.6, 1.0])
# priors = np.array([0.32, 0.02, 0.36, 0.3])
eps = epsilons[np.argmax(ratios)]
# eps = 0.02
priors = np.array([0.32+eps/3, 0.02-eps, 0.36+2*eps/3, 0.3])
print(np.sum(priors))
c = 0.75
# threshold_true = 0.10221
threshold_true = threshold_trues[np.argmax(approx_ratios)]
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.9f}")
print(f"eps:           : {eps:.5f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (benc)       : {2.8936:.4f}")
print(f"r (mult)       : {r:.4f}")
print("-"*80)
print()

### Worst Case Example for n=3

In [ ]:
thresholds = np.array([0., 0.2, 0.6, 1.0])
priors = np.array([0.32, 0.02, 0.36, 0.3])
print(np.sum(priors))
c = 0.75
threshold_true = 0.1
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.9f}\n")
print("Greedy\n------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}\n")
print("Optimal\n-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}\n")
print(f"r (mult)       : {r:.4f}")

### Adding copies of 0

Searching for the optimal value close to 0

In [ ]:
thresholds = np.array([0., 1e-5, 0.2, 0.6, 1.0])
# priors = np.array([0.16, 0.16, 0.02, 0.36, 0.3])
priors = np.array([0.31, 0.01, 0.02, 0.36, 0.3])
c = 0.75
threshold_true = 0.10201
approx_ratios = []

t2s = np.arange(0, 1e-5+1e-7, 1e-7)
for t2 in tqdm.tqdm(t2s):
    thresholds[1] = t2
    partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    approx_ratios.append(r)
approx_ratios = np.array(t2s)

In [ ]:
thresholds = np.array([0., t2s[np.argmax(approx_ratios)], 0.2, 0.6, 1.0])
priors = np.array([0.16, 0.16, 0.02, 0.36, 0.3])
# priors = np.array([0.01, 0.31, 0.02, 0.36, 0.3])
# priors = np.array([0.31, 0.01, 0.02, 0.36, 0.3])
print(np.sum(priors))
c = 0.75
threshold_true = 0.10201
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.9f}\n")
print("Greedy\n------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}\n")
print("Optimal\n-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}\n")
print(f"r (mult)       : {r:.4f}")

### Adding more copies of 0

In [ ]:
thresholds = np.array([0., 1e-7, 1e-5, 0.2, 0.6, 1.0])
# priors = np.array([0.02, 0.15, 0.15, 0.02, 0.36, 0.3])
# priors = np.array([0.15, 0.02, 0.15, 0.02, 0.36, 0.3])
# priors = np.array([0.15, 0.15, 0.02, 0.02, 0.36, 0.3])
priors = np.array([0.3, 0.01, 0.01, 0.02, 0.36, 0.3])
print(np.sum(priors))
c = 0.75
threshold_true = 0.10201
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.9f}\n")
print("Greedy\n------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}\n")
print("Optimal\n-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}\n")
print(f"r (mult)       : {r:.4f}")

In [ ]:
thresholds = np.array([0., 1e-5, 0.2, 0.6, 1.0])
priors = np.array([0.16, 0.16, 0.02, 0.36, 0.3])
print(np.sum(priors))
c = 0.75
threshold_true = threshold_trues[np.argmax(approx_ratios)]
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.9f}\n")
print("Greedy\n------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}\n")
print("Optimal\n-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}\n")
print(f"r (mult)       : {r:.4f}")